# PySpark practice 5: dates, timestamps and time zones

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-05-dates-and-times.ipynb)

15 exercises on parsing, formatting, date arithmetic, time zones, epochs, calendars, buckets and sessions. Each exercise names the [reference entries](pyspark-reference-05-dates-and-times.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 5](pyspark-reference-05-dates-and-times.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

5 easy, 7 medium, 3 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Parse dates arriving in mixed formats](#exercise-1-parse-dates-arriving-in-mixed-formats) | Medium | [`try_to_timestamp`](pyspark-reference-05-dates-and-times.ipynb#try_to_timestamp) |
| 2 | [Business date in the customer's time zone](#exercise-2-business-date-in-the-customers-time-zone) | Medium | [`from_utc_timestamp`, `to_utc_timestamp`](pyspark-reference-05-dates-and-times.ipynb#from_utc_timestamp-to_utc_timestamp) |
| 3 | [Days to ship and SLA breaches](#exercise-3-days-to-ship-and-sla-breaches) | Medium | [`timestamp_diff`, `timestamp_add`](pyspark-reference-05-dates-and-times.ipynb#timestamp_diff-timestamp_add) |
| 4 | [Customer tenure in full months](#exercise-4-customer-tenure-in-full-months) | Easy | [`datediff`, `months_between`](pyspark-reference-05-dates-and-times.ipynb#datediff-months_between) |
| 5 | [Weekly revenue by Monday week start](#exercise-5-weekly-revenue-by-monday-week-start) | Easy | [`date_trunc`, `trunc`](pyspark-reference-05-dates-and-times.ipynb#date_trunc-trunc) |
| 6 | [Month-end and quarter-end dates](#exercise-6-month-end-and-quarter-end-dates) | Medium | [`last_day`, `next_day`](pyspark-reference-05-dates-and-times.ipynb#last_day-next_day), [`date_add`, `date_sub`, `add_months`](pyspark-reference-05-dates-and-times.ipynb#date_add-date_sub-add_months) |
| 7 | [Convert epoch milliseconds from Kafka](#exercise-7-convert-epoch-milliseconds-from-kafka) | Easy | [`timestamp_seconds`, `timestamp_millis`, `unix_seconds`, `unix_millis`](pyspark-reference-05-dates-and-times.ipynb#timestamp_seconds-timestamp_millis-unix_seconds-unix_millis) |
| 8 | [Hourly order counts](#exercise-8-hourly-order-counts) | Easy | [`window`](pyspark-reference-05-dates-and-times.ipynb#window) |
| 9 | [Find days with no orders](#exercise-9-find-days-with-no-orders) | Medium | [Calendars with `sequence`](pyspark-reference-05-dates-and-times.ipynb#calendars-with-sequence) |
| 10 | [Sessions with session_window](#exercise-10-sessions-with-session_window) | Medium | [`session_window`](pyspark-reference-05-dates-and-times.ipynb#session_window) |
| 11 | [Build timestamps from parts in a local zone](#exercise-11-build-timestamps-from-parts-in-a-local-zone) | Hard | [`make_timestamp`, `try_make_timestamp`, `make_timestamp_ntz`](pyspark-reference-05-dates-and-times.ipynb#make_timestamp-try_make_timestamp-make_timestamp_ntz) |
| 12 | [Local wall-clock across a DST change](#exercise-12-local-wall-clock-across-a-dst-change) | Hard | [`from_utc_timestamp`, `to_utc_timestamp`](pyspark-reference-05-dates-and-times.ipynb#from_utc_timestamp-to_utc_timestamp) |
| 13 | [Expand date ranges into one row per day](#exercise-13-expand-date-ranges-into-one-row-per-day) | Hard | [Calendars with `sequence`](pyspark-reference-05-dates-and-times.ipynb#calendars-with-sequence) |
| 14 | [Format dates for an export contract](#exercise-14-format-dates-for-an-export-contract) | Easy | [Pattern letters](pyspark-reference-05-dates-and-times.ipynb#pattern-letters), [`date_format`](pyspark-reference-05-dates-and-times.ipynb#date_format) |
| 15 | [Time since the previous order with intervals](#exercise-15-time-since-the-previous-order-with-intervals) | Medium | [Intervals: `make_interval`, `make_dt_interval` and interval arithmetic](pyspark-reference-05-dates-and-times.ipynb#intervals-make_interval-make_dt_interval-and-interval-arithmetic) |

## Exercise 1: Parse dates arriving in mixed formats

**Difficulty:** Medium · about 10 min · **Practises:** [`try_to_timestamp`](pyspark-reference-05-dates-and-times.ipynb#try_to_timestamp)

*Feeds from several systems disagree on date formats; invalid dates must surface, not crash the job.*

Parse `text` trying `yyyy-MM-dd`, then `dd/MM/yyyy`, then `MMM d, yyyy`. Return `text` and `parsed` (a date, null when no format fits). Keep input order.

In [ ]:
raw = spark.createDataFrame([(1, "2026-01-31"), (2, "31/01/2026"), (3, "Jan 5, 2026"), (4, "2026-02-30"), (5, None)], "n int, text string")
raw.show()

**Expected output**

| text | parsed |
| --- | --- |
| 2026-01-31 | 2026-01-31 |
| 31/01/2026 | 2026-01-31 |
| Jan 5, 2026 | 2026-01-05 |
| 2026-02-30 | null |
| null | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
formats = ["yyyy-MM-dd", "dd/MM/yyyy", "MMM d, yyyy"]
parsed = F.to_date(F.coalesce(*[F.try_to_timestamp("text", F.lit(f)) for f in formats]))
result = raw.orderBy("n").select("text", parsed.alias("parsed"))
result.show()

## Exercise 2: Business date in the customer's time zone

**Difficulty:** Medium · about 10 min · **Practises:** [`from_utc_timestamp`, `to_utc_timestamp`](pyspark-reference-05-dates-and-times.ipynb#from_utc_timestamp-to_utc_timestamp)

*An order just after midnight UTC belongs to the previous day in the Americas.*

Each customer region has a time zone in `zones`. Return `order_id`, `utc_date` and `local_date` (the order date in the customer's zone; UTC when the region is unknown) for orders 8 to 12, ordered by `order_id`.

In [ ]:
zones = spark.createDataFrame([("North", "America/New_York"), ("South", "America/Chicago"), ("West", "America/Los_Angeles"), ("East", "Europe/Berlin")], "region string, tz string")
zones.show()

**Expected output**

| order_id | utc_date | local_date |
| --- | --- | --- |
| 8 | 2026-01-10 | 2026-01-10 |
| 9 | 2026-01-12 | 2026-01-12 |
| 10 | 2026-01-12 | 2026-01-12 |
| 11 | 2026-01-13 | 2026-01-13 |
| 12 | 2026-01-14 | 2026-01-13 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
tz = F.coalesce("tz", F.lit("UTC"))
result = (
    orders.where("order_id BETWEEN 8 AND 12")
    .join(customers.select("customer_id", "region"), "customer_id", "left")
    .join(zones, "region", "left")
    .select("order_id", F.to_date("order_ts").alias("utc_date"), F.to_date(F.from_utc_timestamp("order_ts", tz)).alias("local_date"))
    .orderBy("order_id")
)
result.show()

## Exercise 3: Days to ship and SLA breaches

**Difficulty:** Medium · about 10 min · **Practises:** [`timestamp_diff`, `timestamp_add`](pyspark-reference-05-dates-and-times.ipynb#timestamp_diff-timestamp_add)

*Fulfilment SLAs are measured in whole hours from placement to shipment.*

Return `order_id`, `hours_to_ship` (whole hours) and `breached` (more than 48 hours), ordered by `order_id`.

In [ ]:
fulfilment = spark.createDataFrame([(1, datetime(2026, 1, 5, 9, 0), datetime(2026, 1, 6, 8, 0)), (2, datetime(2026, 1, 5, 9, 0), datetime(2026, 1, 7, 10, 30)),
                                    (3, datetime(2026, 1, 6, 23, 0), datetime(2026, 1, 8, 22, 59))], "order_id int, placed timestamp, shipped timestamp")
fulfilment.show()

**Expected output**

| order_id | hours_to_ship | breached |
| --- | --- | --- |
| 1 | 23 | false |
| 2 | 49 | true |
| 3 | 47 | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
hours = F.timestamp_diff("HOUR", "placed", "shipped")
result = fulfilment.select("order_id", hours.alias("hours_to_ship"), (hours > 48).alias("breached")).orderBy("order_id")
result.show()

## Exercise 4: Customer tenure in full months

**Difficulty:** Easy · about 5 min · **Practises:** [`datediff`, `months_between`](pyspark-reference-05-dates-and-times.ipynb#datediff-months_between)

*Cohort analysis buckets customers by tenure.*

As of 2026-01-31, return `customer_id`, `tenure_days` and `tenure_months` (whole months, rounded down), ordered by `customer_id`.

In [ ]:
customers.select("customer_id", "signup_date").show()

**Expected output**

| customer_id | tenure_days | tenure_months |
| --- | --- | --- |
| 1 | 759 | 24 |
| 2 | 720 | 23 |
| 3 | 687 | 22 |
| 4 | 639 | 20 |
| 5 | 590 | 19 |
| 6 | 548 | 17 |
| 7 | 504 | 16 |
| 8 | 452 | 14 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
as_of = F.lit("2026-01-31").cast("date")
result = customers.select("customer_id", F.datediff(as_of, "signup_date").alias("tenure_days"),
                          F.floor(F.months_between(as_of, "signup_date")).cast("int").alias("tenure_months")).orderBy("customer_id")
result.show()

## Exercise 5: Weekly revenue by Monday week start

**Difficulty:** Easy · about 5 min · **Practises:** [`date_trunc`, `trunc`](pyspark-reference-05-dates-and-times.ipynb#date_trunc-trunc)

*Weekly reports group by a sortable week-start date, not a week number.*

Return `week_start` (date of the Monday starting each order's week) and `revenue`, ordered by `week_start`.

In [ ]:
orders.select("order_id", "order_ts", "amount").show()

**Expected output**

| week_start | revenue |
| --- | --- |
| 2026-01-05 | 1098.65 |
| 2026-01-12 | 1560.40 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.groupBy(F.to_date(F.date_trunc("week", "order_ts")).alias("week_start")).agg(F.sum("amount").alias("revenue")).orderBy("week_start")
result.show()

## Exercise 6: Month-end and quarter-end dates

**Difficulty:** Medium · about 10 min · **Practises:** [`last_day`, `next_day`](pyspark-reference-05-dates-and-times.ipynb#last_day-next_day), [`date_add`, `date_sub`, `add_months`](pyspark-reference-05-dates-and-times.ipynb#date_add-date_sub-add_months)

*Accruals and reporting cut-offs are month-end and quarter-end dates.*

For each date in `dates`, return `d`, `month_end`, `quarter_end` and `next_monday`. Keep input order.

In [ ]:
dates = spark.createDataFrame([(1, date(2026, 1, 31)), (2, date(2026, 2, 14)), (3, date(2026, 5, 4))], "n int, d date")
dates.show()

**Expected output**

| d | month_end | quarter_end | next_monday |
| --- | --- | --- | --- |
| 2026-01-31 | 2026-01-31 | 2026-03-31 | 2026-02-02 |
| 2026-02-14 | 2026-02-28 | 2026-03-31 | 2026-02-16 |
| 2026-05-04 | 2026-05-31 | 2026-06-30 | 2026-05-11 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
quarter_end = F.date_sub(F.add_months(F.trunc("d", "quarter"), 3), 1)
result = dates.orderBy("n").select("d", F.last_day("d").alias("month_end"), quarter_end.alias("quarter_end"), F.next_day("d", "Mon").alias("next_monday"))
result.show()

## Exercise 7: Convert epoch milliseconds from Kafka

**Difficulty:** Easy · about 5 min · **Practises:** [`timestamp_seconds`, `timestamp_millis`, `unix_seconds`, `unix_millis`](pyspark-reference-05-dates-and-times.ipynb#timestamp_seconds-timestamp_millis-unix_seconds-unix_millis)

*Message brokers and JavaScript clients send epoch milliseconds.*

Return `event_id`, `event_time` (timestamp from `ts_ms`) and `lag_seconds` (seconds between `event_time` and `received_at`), ordered by `event_id`.

In [ ]:
msgs = spark.createDataFrame([(1, 1768350600000, datetime(2026, 1, 14, 0, 30, 5)), (2, 1768350660500, datetime(2026, 1, 14, 0, 31, 30))],
                             "event_id int, ts_ms bigint, received_at timestamp")
msgs.show()

**Expected output**

| event_id | event_time | lag_seconds |
| --- | --- | --- |
| 1 | 2026-01-14 00:30:00 | 5.0 |
| 2 | 2026-01-14 00:31:00.500000 | 29.5 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
event_time = F.timestamp_millis("ts_ms")
result = msgs.select("event_id", event_time.alias("event_time"),
                     ((F.unix_millis("received_at") - F.col("ts_ms")) / 1000).alias("lag_seconds")).orderBy("event_id")
result.show(truncate=False)

## Exercise 8: Hourly order counts

**Difficulty:** Easy · about 5 min · **Practises:** [`window`](pyspark-reference-05-dates-and-times.ipynb#window)

*Operational dashboards plot counts per fixed time bucket.*

Count orders per 6-hour window. Return `start`, `end` and `orders` for windows with at least one order, ordered by `start`.

In [ ]:
orders.select("order_id", "order_ts").show()

**Expected output**

| start | end | orders |
| --- | --- | --- |
| 2026-01-05 06:00:00 | 2026-01-05 12:00:00 | 3 |
| 2026-01-06 12:00:00 | 2026-01-06 18:00:00 | 1 |
| 2026-01-07 06:00:00 | 2026-01-07 12:00:00 | 1 |
| 2026-01-07 12:00:00 | 2026-01-07 18:00:00 | 1 |
| 2026-01-09 12:00:00 | 2026-01-09 18:00:00 | 1 |
| 2026-01-10 18:00:00 | 2026-01-11 00:00:00 | 1 |
| 2026-01-12 06:00:00 | 2026-01-12 12:00:00 | 1 |
| 2026-01-12 12:00:00 | 2026-01-12 18:00:00 | 1 |
| 2026-01-13 06:00:00 | 2026-01-13 12:00:00 | 1 |
| 2026-01-14 00:00:00 | 2026-01-14 06:00:00 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.groupBy(F.window("order_ts", "6 hours")).agg(F.count("*").alias("orders")).select("window.start", "window.end", "orders").orderBy("start")
result.show()

## Exercise 9: Find days with no orders

**Difficulty:** Medium · about 10 min · **Practises:** [Calendars with `sequence`](pyspark-reference-05-dates-and-times.ipynb#calendars-with-sequence)

*Gaps in a daily feed usually mean a failed load, not a quiet day.*

Between the first and last order date (UTC), return every `day` with no orders, ordered.

In [ ]:
orders.select(F.to_date("order_ts").alias("day")).distinct().orderBy("day").show()

**Expected output**

| day |
| --- |
| 2026-01-08 |
| 2026-01-11 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
bounds = orders.agg(F.min(F.to_date("order_ts")).alias("lo"), F.max(F.to_date("order_ts")).alias("hi"))
calendar = bounds.select(F.explode(F.sequence("lo", "hi")).alias("day"))
active = orders.select(F.to_date("order_ts").alias("day")).distinct()
result = calendar.join(active, "day", "left_anti").orderBy("day")
result.show()

## Exercise 10: Sessions with session_window

**Difficulty:** Medium · about 10 min · **Practises:** [`session_window`](pyspark-reference-05-dates-and-times.ipynb#session_window)

*Session metrics per user without hand-written window logic.*

After removing duplicate events, group each user's events into sessions with a 1-hour gap. Return `user_id`, `session_start`, `events`, ordered by `user_id`, `session_start`.

In [ ]:
events.orderBy("user_id", "ts").show()

**Expected output**

| user_id | session_start | events |
| --- | --- | --- |
| 1 | 2026-01-06 09:00:00 | 3 |
| 2 | 2026-01-07 10:00:00 | 2 |
| 3 | 2026-01-06 11:15:00 | 1 |
| 3 | 2026-01-08 11:20:00 | 1 |
| 4 | 2026-01-09 20:05:00 | 1 |
| 4 | 2026-01-10 08:30:00 | 1 |
| 5 | 2026-01-11 14:45:00 | 1 |
| 5 | 2026-01-12 09:00:00 | 1 |
| 5 | 2026-01-12 12:30:00 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    events.dropDuplicates()
    .groupBy("user_id", F.session_window("ts", "1 hour").alias("s"))
    .agg(F.count("*").alias("events"))
    .select("user_id", F.col("s.start").alias("session_start"), "events")
    .orderBy("user_id", "session_start")
)
result.show()

## Exercise 11: Build timestamps from parts in a local zone

**Difficulty:** Hard · about 15 min · **Practises:** [`make_timestamp`, `try_make_timestamp`, `make_timestamp_ntz`](pyspark-reference-05-dates-and-times.ipynb#make_timestamp-try_make_timestamp-make_timestamp_ntz)

*Legacy systems store local date and time parts in separate integer columns.*

Build the instant for each row from its parts, interpreting them in `tz`. Return `id` and `utc_text` (the instant formatted `yyyy-MM-dd HH:mm` in UTC), null for impossible dates. Order by `id`.

In [ ]:
parts = spark.createDataFrame([(1, 2026, 1, 13, 18, 30, "America/Chicago"), (2, 2026, 1, 14, 9, 0, "Asia/Kolkata"), (3, 2026, 2, 30, 9, 0, "UTC")],
                              "id int, y int, mo int, d int, h int, mi int, tz string")
parts.show()

**Expected output**

| id | utc_text |
| --- | --- |
| 1 | 2026-01-14 00:30 |
| 2 | 2026-01-14 03:30 |
| 3 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
instant = F.try_make_timestamp("y", "mo", "d", "h", "mi", F.lit(0), F.col("tz"))
result = parts.select("id", F.date_format(instant, "yyyy-MM-dd HH:mm").alias("utc_text")).orderBy("id")
result.show()

## Exercise 12: Local wall-clock across a DST change

**Difficulty:** Hard · about 15 min · **Practises:** [`from_utc_timestamp`, `to_utc_timestamp`](pyspark-reference-05-dates-and-times.ipynb#from_utc_timestamp-to_utc_timestamp)

*Fixed offsets break twice a year; named zones handle daylight saving time.*

For each UTC instant in `instants`, return `utc` and `new_york` (wall-clock time in America/New_York) as strings `yyyy-MM-dd HH:mm`, in input order. US daylight saving starts on 2026-03-08 at 02:00 local time.

In [ ]:
instants = spark.createDataFrame([(1, datetime(2026, 3, 8, 6, 30)), (2, datetime(2026, 3, 8, 7, 30)), (3, datetime(2026, 7, 1, 12, 0))], "n int, ts timestamp")
instants.show()

**Expected output**

| utc | new_york |
| --- | --- |
| 2026-03-08 06:30 | 2026-03-08 01:30 |
| 2026-03-08 07:30 | 2026-03-08 03:30 |
| 2026-07-01 12:00 | 2026-07-01 08:00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
fmt = "yyyy-MM-dd HH:mm"
result = instants.orderBy("n").select(F.date_format("ts", fmt).alias("utc"), F.date_format(F.from_utc_timestamp("ts", "America/New_York"), fmt).alias("new_york"))
result.show()

## Exercise 13: Expand date ranges into one row per day

**Difficulty:** Hard · about 15 min · **Practises:** [Calendars with `sequence`](pyspark-reference-05-dates-and-times.ipynb#calendars-with-sequence)

*Bookings, promotions and absences are stored as ranges but reported per day.*

Expand each booking into one row per night (`check_in` inclusive, `check_out` exclusive). Return `room` and `nights_booked` per room for January 2026 only, ordered by `room`.

In [ ]:
bookings = spark.createDataFrame([("101", date(2025, 12, 30), date(2026, 1, 3)), ("101", date(2026, 1, 10), date(2026, 1, 12)), ("102", date(2026, 1, 30), date(2026, 2, 2))],
                                 "room string, check_in date, check_out date")
bookings.show()

**Expected output**

| room | nights_booked |
| --- | --- |
| 101 | 4 |
| 102 | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
nights = bookings.select("room", F.explode(F.sequence("check_in", F.date_sub("check_out", 1))).alias("night"))
result = nights.where(F.col("night").between(F.lit("2026-01-01").cast("date"), F.lit("2026-01-31").cast("date"))).groupBy("room").agg(F.count("*").alias("nights_booked")).orderBy("room")
result.show()

## Exercise 14: Format dates for an export contract

**Difficulty:** Easy · about 5 min · **Practises:** [Pattern letters](pyspark-reference-05-dates-and-times.ipynb#pattern-letters), [`date_format`](pyspark-reference-05-dates-and-times.ipynb#date_format)

*Downstream systems specify exact text formats.*

Return `order_id`, `iso` (`yyyy-MM-dd'T'HH:mm:ss'Z'`), `us` (`MM/dd/yyyy`) and `label` (`EEE d MMM`) for orders 1 and 12, ordered by `order_id`.

In [ ]:
orders.where("order_id IN (1, 12)").select("order_id", "order_ts").show()

**Expected output**

| order_id | iso | us | label |
| --- | --- | --- | --- |
| 1 | 2026-01-05T09:15:00Z | 01/05/2026 | Mon 5 Jan |
| 12 | 2026-01-14T00:30:00Z | 01/14/2026 | Wed 14 Jan |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.where("order_id IN (1, 12)").select(
    "order_id", F.date_format("order_ts", "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("iso"), F.date_format("order_ts", "MM/dd/yyyy").alias("us"),
    F.date_format("order_ts", "EEE d MMM").alias("label")).orderBy("order_id")
result.show()

## Exercise 15: Time since the previous order with intervals

**Difficulty:** Medium · about 10 min · **Practises:** [Intervals: `make_interval`, `make_dt_interval` and interval arithmetic](pyspark-reference-05-dates-and-times.ipynb#intervals-make_interval-make_dt_interval-and-interval-arithmetic)

*Repeat-purchase analysis measures gaps between orders.*

For customers 1 to 3, return `order_id` and `minutes_since_prev` (whole minutes since the same customer's previous order, null for the first). Order by `order_id`.

In [ ]:
orders.where("customer_id <= 3").select("order_id", "customer_id", "order_ts").show()

**Expected output**

| order_id | minutes_since_prev |
| --- | --- |
| 1 | null |
| 2 | 50 |
| 3 | 95 |
| 4 | null |
| 5 | 1005 |
| 6 | null |
| 7 | 2545 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
prev = F.lag("order_ts").over(Window.partitionBy("customer_id").orderBy("order_ts"))
result = orders.where("customer_id <= 3").select("order_id", F.timestamp_diff("MINUTE", prev, F.col("order_ts")).alias("minutes_since_prev")).orderBy("order_id")
result.show()

**Next:** look anything up in [PySpark reference 5: dates, timestamps and time zones](pyspark-reference-05-dates-and-times.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).